In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


Here, I will extract the **k20 model** used in the Kobayashi, Karakas and Lugaro (2020) paper (https://doi.org/10.3847/1538-4357/abae65). The data this GCE model, alongside every other done by Kobayashi are all found on this database: https://star.herts.ac.uk/~chiaki/gce/. 

Instructions on how to read each file is at the bottom of the link, but for k20 (solar neighbourhood), it has the following columns:
1. **Time** (0 to 15Gyr in 100Myr steps)
2. **X/X_sun** (where X is all stable elements ordered by atomic number. E.g. H first, then He, then Li etc)

Since the elements are **not logarithmic**, they need to be converted to the square bracket notation.

**For [Fe/H]:**
1. Perform the following calculation: $$[Fe/H] = log(Fe_{col}) - log(H_{col})$$
whereby $$Fe_{col} = Fe/Fe_{\odot}$$ $$H_{col} = H/H_{\odot}$$ 

**For [X/Fe]:**
1. Perform the following calculation: $$[X/Fe] = log(X_{col}) - log(Fe_{col})$$
whereby $$X_{col} = X/X_{\odot}$$ $$Fe_{col} = Fe/Fe_{\odot}$$

In [14]:
# Loading in the dataset
# NOTE: 
# col 0 is time,
# col 1 is H/H_sun 
# col 2 is He/H_sun
# etc etc
sz_df = pd.read_csv("/data/misfit/benlowe/xshooter/p115/gce_models/sz_mw/sz_mw_gce.dat", delim_whitespace=True, header=None)
hz_df = pd.read_csv("/data/misfit/benlowe/xshooter/p115/gce_models/hz_mw/hz_mw_gce.dat", delim_whitespace=True, header=None)
hzto03_df = pd.read_csv("/data/misfit/benlowe/xshooter/p115/gce_models/hzto03_mw/hzto03_mw_gce.dat", delim_whitespace=True, header=None)
wtz_df = pd.read_csv("/data/misfit/benlowe/xshooter/p115/gce_models/wtz_mw/wtz_mw_gce.dat", delim_whitespace=True, header=None)
wzSclsx50_df = pd.read_csv("/data/misfit/benlowe/xshooter/p115/gce_models/wzSclsx50_Sculp/wzSclsx50_sculp_gce.dat", delim_whitespace=True, header=None)

# List of elements we want (element symbol and their column number)
# Note that 43Tc and 61Pm are not included in the dataset, so need to adjust the column numbers accordingly
els = {"C": 6, 
       "N": 7, 
       "Na": 11,
       "Mg": 12,
       "Al": 13, 
       "Si": 14, 
       "Ca": 20, 
       "Sc": 21, 
       "Ti": 21,
       "Cr": 24, 
       "Mn": 25, 
       "Co": 27, 
       "Ni": 28, 
       "Sr": 38, 
       "Ba": 55, 
       "Eu": 61}

/tmp/ipykernel_2746134/163751054.py:7: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  sz_df = pd.read_csv("/data/misfit/benlowe/xshooter/p115/gce_models/sz_mw/sz_mw_gce.dat", delim_whitespace=True, header=None)
/tmp/ipykernel_2746134/163751054.py:8: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  hz_df = pd.read_csv("/data/misfit/benlowe/xshooter/p115/gce_models/hz_mw/hz_mw_gce.dat", delim_whitespace=True, header=None)
/tmp/ipykernel_2746134/163751054.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  hzto03_df = pd.read_csv("/data/misfit/benlowe/xshooter/p115/gce_models/hzto03_mw/hzto03_mw_gce.dat", delim_whitespace=True, header=None)
/tmp/ipykernel_2746134/163751054.py:10: FutureWarning: The 'deli

In [16]:
def extract_gce_abunds(gce_df, els_dict, savepath):
    
    # Getting metallicity [Fe/H] from the GCE dataframe
    # Doing this by: [Fe/H] = log10(Fe_entry) - log10(H_entry)
    fe_entry = gce_df.iloc[:, 26].values
    h_entry = gce_df.iloc[:, 1].values
    feh = np.log10(fe_entry) - np.log10(h_entry)
    
    # Loop over the element dictionary
    for el, col in els_dict.items():
        
        # Extracting the element entry from the GCE dataframe based on its column number, 
        # alongside the Fe entry (col 26)
        el_entry = gce_df.iloc[:, col].values
        
        # Getting the [X/Fe] abundance
        # Doing this by: [X/Fe] = log10(el_entry) - log10(fe_entry)
        xfe = np.log10(el_entry) - np.log10(fe_entry)
        
        # Storing the [Fe/H] and [X/Fe] abundances into a dataframe, then saving as a csv file
        el_abunds_df = pd.DataFrame({"feh": feh, f"{el.lower()}fe": xfe})
        el_abunds_df.to_csv(f"{savepath}/{el.lower()}fe_abunds.csv", index=False)
        

# Extracting, then saving the abundances for each element in the dictionary
sz_savepath = "/data/misfit/benlowe/xshooter/p115/gce_models/sz_mw"
extract_gce_abunds(gce_df = sz_df, 
                   els_dict = els,
                   savepath = sz_savepath)

hz_savepath = "/data/misfit/benlowe/xshooter/p115/gce_models/hz_mw"
extract_gce_abunds(gce_df = hz_df,
                   els_dict = els,
                   savepath = hz_savepath)

hzto03_savepath = "/data/misfit/benlowe/xshooter/p115/gce_models/hzto03_mw"
extract_gce_abunds(gce_df = hzto03_df,
                   els_dict = els,
                   savepath = hzto03_savepath)

wtz_savepath = "/data/misfit/benlowe/xshooter/p115/gce_models/wtz_mw"
extract_gce_abunds(gce_df = wtz_df,
                   els_dict = els,
                   savepath = wtz_savepath)

wzSclsx50_savepath = "/data/misfit/benlowe/xshooter/p115/gce_models/wzSclsx50_Sculp"
extract_gce_abunds(gce_df = wzSclsx50_df,
                   els_dict = els,
                   savepath = wzSclsx50_savepath)

/tmp/ipykernel_2746134/1023898554.py:7: RuntimeWarning: divide by zero encountered in log10
  feh = np.log10(fe_entry) - np.log10(h_entry)
/tmp/ipykernel_2746134/1023898554.py:18: RuntimeWarning: divide by zero encountered in log10
  xfe = np.log10(el_entry) - np.log10(fe_entry)
/tmp/ipykernel_2746134/1023898554.py:18: RuntimeWarning: invalid value encountered in subtract
  xfe = np.log10(el_entry) - np.log10(fe_entry)
